# Project 62 — Local Output Judge
## Model-A Output → Model-B Critique → Structured Scoring

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

## Step 1 — Generate Candidate Outputs

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

llm = ChatOllama(model="qwen3:8b", temperature=0.7)

tasks = [
    "Write a function to check if a string is a palindrome",
    "Explain quantum entanglement to a 10-year-old",
    "Draft a professional email declining a meeting invitation",
]

# Generate multiple candidate outputs per task
candidates = {}
for task in tasks:
    outputs = []
    for temp in [0.1, 0.5, 0.9]:
        gen = ChatOllama(model="qwen3:8b", temperature=temp)
        chain = ChatPromptTemplate.from_template("{task}") | gen | StrOutputParser()
        outputs.append(chain.invoke({"task": task}))
    candidates[task] = outputs
    print(f"Generated {len(outputs)} candidates for: {task[:40]}...")

## Step 2 — Build the Judge

In [ ]:
class Judgment(BaseModel):
    relevance: int = Field(ge=1, le=5, description="How relevant to the task")
    quality: int = Field(ge=1, le=5, description="Overall quality")
    issues: list[str] = Field(description="Problems found")
    verdict: str = Field(description="pass or fail")

judge = ChatOllama(model="qwen3:8b", temperature=0.0).with_structured_output(Judgment)

all_judgments = []
for task, outputs in candidates.items():
    print(f"\nJudging: {task[:40]}...")
    for i, output in enumerate(outputs):
        j = judge.invoke(f"Task: {task}\n\nOutput to judge:\n{output[:500]}")
        j_dict = j.model_dump()
        j_dict["task"] = task[:40]
        j_dict["candidate"] = i + 1
        all_judgments.append(j_dict)
        print(f"  Candidate {i+1}: relevance={j.relevance} quality={j.quality} verdict={j.verdict}")
        if j.issues:
            print(f"    Issues: {j.issues}")

## Step 3 — Aggregate Scores

In [ ]:
import pandas as pd

jdf = pd.DataFrame(all_judgments)
print("\nJUDGMENT SUMMARY")
print("="*60)
print(jdf[["task","candidate","relevance","quality","verdict"]].to_string(index=False))

print(f"\nPass rate: {(jdf['verdict']=='pass').mean()*100:.0f}%")
print(f"Average quality: {jdf['quality'].mean():.1f}/5")

## What You Learned
- **LLM-as-judge** pattern with structured output
- **Multi-temperature generation** for diversity
- **Quantitative quality assessment** of LLM outputs